In [15]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, xgboost as xgb
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
print('工作目录:', os.getcwd())

工作目录: d:\github\prediction


In [ ]:
# ============================================================
# 数据: NetCDF 9站 → 80/15/5 分割 → 预处理 → 全局直接多步特征
# 特征构成 (单模型, horizon 为特征):
#  HIST@T  : 目标站 PM滞后/滚动/差分/峰值 + 8个其他站 PM(t-1, r6m)
#            + 额外气象(blh/msdwswrf/O3) 历史(t-1, m6)
#  FUT@(T+h-1): 目标站 气象(DEWP..precip) + 额外气象(blh/swr/O3) + 时间/风向
#  + horizon h
# 目标 = 线性 PM@(T+h-1); 最近观测 pm[T-1], h=1→pm[T]
# 说明: 9站同属上海集群(corr 0.93-0.98), 多站PM提供空间上下文;
#       blh(边界层高度)决定扩散能力, 对峰值预测尤为关键
# 参考: https://xgboost.readthedocs.io/en/stable/python/python_api.html
# ============================================================
import netCDF4 as nc

N_ST = 9   # NetCDF 站点数 (上海集群: 普陀/十五厂/虹口/徐汇/杨浦/静安/浦东川沙/浦东新区/浦东张江)
WEATHER = ['DEWP','HUMI','PRES','TEMP','Iws','precipitation']
WEATHER_M6 = [f'{c}_m6' for c in WEATHER]
EXTRA = ['blh','msdwswrf','O3']                 # 边界层高度/短波辐射/臭氧
EXTRA_M6 = [f'{c}_m6' for c in EXTRA]
EXTRA_H  = [f'{c}_h' for c in EXTRA]            # 历史侧 (t-1), 避免与 FUT 重名
EXTRA_H_M6 = [f'{c}_h_m6' for c in EXTRA]
PM_LAGS = [1,2,3,6,12,24]; MAX_LAG = 24
TIME_COLS = ['hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos']
CBWD_COLS = ['cbwd_cv','cbwd_SE','cbwd_NW','cbwd_SW','cbwd_NE']
ROLL_COLS = ['pm_r6m','pm_r6s','pm_r24m','pm_r24s']
OTHER_COLS = [f'pm{i}_t1' for i in range(1,N_ST)] + [f'pm{i}_r6m' for i in range(1,N_ST)]  # 8站PM历史
HIST_COLS = ([f'pm_t-{l}' for l in PM_LAGS] + ROLL_COLS +
             ['pm_d_short','pm_d_long','pm_r6x','pm_r24x','precip_r24'] +
             OTHER_COLS + EXTRA_H + EXTRA_H_M6)
FUT_COLS  = WEATHER + WEATHER_M6 + TIME_COLS + CBWD_COLS + EXTRA + EXTRA_M6
FEAT = HIST_COLS + FUT_COLS + ['horizon']
NH, NF, N_FEAT = len(HIST_COLS), len(FUT_COLS), len(FEAT)

def load_all(nc_path):
    """读取 9站 PM2.5 + 站0 全部气象/额外变量 (变量映射: K→°C, Pa→hPa, m→mm)"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    pm = np.asarray(f.variables['PM2.5'][:])          # (T,9) 全部9站 PM
    temp = f.variables['t2m'][:,0]-273.15; dewp = f.variables['d2m'][:,0]-273.15
    pres = f.variables['sp'][:,0]/100.0;  tp = f.variables['tp'][:,0]*1000.0
    u, v = f.variables['u100'][:,0], f.variables['v100'][:,0]
    iws = np.sqrt(u**2+v**2); wdir = np.degrees(np.arctan2(-u,-v))%360
    cbwd = np.where(iws<0.5,'cv',np.where(wdir<90,'NE',np.where(wdir<180,'SE',np.where(wdir<270,'SW','NW')))).astype('<U2')
    a,b = 17.625,243.04
    humi = np.clip(100*np.exp(a*dewp/(dewp+b+1e-10))/np.exp(a*temp/(temp+b+1e-10)),0,100)
    blh = f.variables['blh'][:,0]; swr = f.variables['msdwswrf'][:,0]; o3 = f.variables['O3'][:,0]
    f.close()
    d = {'pm_ave': pm[:,0]}                            # 目标站 PM
    for i in range(1,N_ST): d[f'pm{i}'] = pm[:,i]     # 其他8站 PM
    d.update({'DEWP':dewp,'HUMI':humi,'PRES':pres,'TEMP':temp,'cbwd':cbwd,'Iws':iws,
              'precipitation':tp,'blh':blh,'msdwswrf':swr,'O3':o3})
    df = pd.DataFrame(d, index=dt); df.index.name='datetime'
    return df.resample('1h').first()

def preprocess(df, forward_only=False):
    """预处理. forward_only=True(测试集)仅前向填充, 不用任何未来值."""
    d = df.copy()
    def fill(s): return s.ffill() if forward_only else s.interpolate(method='time',limit=168).ffill().bfill()
    d['pm_ave'] = np.log1p(fill(d['pm_ave']))          # 目标站 PM (log 稳定特征)
    for i in range(1,N_ST): d[f'pm{i}'] = np.log1p(fill(d[f'pm{i}']))   # 其他站 PM
    cols = WEATHER + EXTRA
    d[cols] = d[cols].ffill() if forward_only else d[cols].interpolate(method='time').ffill().bfill()
    h = d.index.hour.values.astype(float)
    d['hour_sin']=np.sin(2*np.pi*h/24); d['hour_cos']=np.cos(2*np.pi*h/24)
    m = d.index.month.values.astype(float)
    d['month_sin']=np.sin(2*np.pi*m/12); d['month_cos']=np.cos(2*np.pi*m/12)
    dw = d.index.dayofweek.values.astype(float)
    d['dow_sin']=np.sin(2*np.pi*dw/7); d['dow_cos']=np.cos(2*np.pi*dw/7)
    d['cbwd'] = d['cbwd'].ffill().bfill()
    d['cbwd'] = pd.Categorical(d['cbwd'], categories=['cv','SE','NW','SW','NE'])
    d = pd.concat([d.drop('cbwd',axis=1), pd.get_dummies(d['cbwd'],prefix='cbwd').astype(float)], axis=1)
    d['pm_r6m']=d['pm_ave'].rolling(6,min_periods=1).mean();  d['pm_r6s']=d['pm_ave'].rolling(6,min_periods=1).std().fillna(0)
    d['pm_r24m']=d['pm_ave'].rolling(24,min_periods=1).mean(); d['pm_r24s']=d['pm_ave'].rolling(24,min_periods=1).std().fillna(0)
    for c in WEATHER: d[f'{c}_m6']=d[c].rolling(6,min_periods=1).mean()
    for c in EXTRA:   d[f'{c}_m6']=d[c].rolling(6,min_periods=1).mean()
    d['precip_r24']=d['precipitation'].rolling(24,min_periods=1).sum()
    return d

def hist_matrix(df):
    """历史特征矩阵 (行 t 仅用 t-1 及更早, 无泄漏). 返回 (n, NH) ndarray."""
    pm = df['pm_ave']; c = {}
    for l in PM_LAGS: c[f'pm_t-{l}'] = pm.shift(l)
    for col in ROLL_COLS: c[col] = df[col].shift(1)
    c['pm_d_short']=pm.shift(1)-pm.shift(3); c['pm_d_long']=pm.shift(1)-pm.shift(24)
    c['pm_r6x']=pm.rolling(6,min_periods=1).max().shift(1); c['pm_r24x']=pm.rolling(24,min_periods=1).max().shift(1)
    c['precip_r24']=df['precip_r24'].shift(1)
    for i in range(1,N_ST):
        c[f'pm{i}_t1']=df[f'pm{i}'].shift(1); c[f'pm{i}_r6m']=df[f'pm{i}'].rolling(6,min_periods=1).mean().shift(1)
    for e in EXTRA:
        c[f'{e}_h']=df[e].shift(1); c[f'{e}_h_m6']=df[f'{e}_m6'].shift(1)
    return pd.DataFrame(c, index=df.index).values

def build_dataset(df, horizons):
    """全局直接样本: 行(T,h)=hist@T + fut@(T+h-1) + h, 目标=线性 PM@(T+h-1)."""
    hist = hist_matrix(df); fut = df[FUT_COLS].values; ylin = np.expm1(df['pm_ave'].values)
    n = len(df); Xs, ys = [], []
    for h in horizons:
        i0, i1 = MAX_LAG, n - h
        if i1 <= i0: continue
        cnt = i1 - i0
        Xh = np.empty((cnt, N_FEAT))
        Xh[:, :NH] = hist[i0:i1]
        Xh[:, NH:NH+NF] = fut[i0+h-1:i1+h-1]
        Xh[:, NH+NF] = h
        Xs.append(Xh); ys.append(ylin[i0+h-1:i1+h-1])
    return np.vstack(Xs), np.concatenate(ys)

# 加载 + 分割 + 预处理 (先分割再处理, 杜绝泄漏; 测试集前向填充)
df_raw = load_all('data3/dataset_yrd.nc')
n = len(df_raw); n1 = int(n*0.80); n2 = int(n*0.95)
df_tr = preprocess(df_raw.iloc[:n1])
df_va = preprocess(df_raw.iloc[n1:n2])
df_te = preprocess(df_raw.iloc[n2:], forward_only=True)
df_va_ctx = pd.concat([df_tr.iloc[-MAX_LAG:], df_va])
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)} | 特征 {N_FEAT} (含8站PM+blh/swr/O3)')

X_tr, y_tr = build_dataset(df_tr, list(range(1, 49)))
X_va, y_va = build_dataset(df_va_ctx, list(range(1, 49, 6)))
print(f'全局样本: 训练 {X_tr.shape} | 验证 {X_va.shape}')

CLIP_HI = float(np.expm1(df_tr['pm_ave'].max()))
print(f'预测上限 {CLIP_HI:.0f} | 训练目标 max={y_tr.max():.0f} p99={np.quantile(y_tr,0.99):.0f} 中位={np.median(y_tr):.0f}')

In [ ]:
# ============================================================
# XGBoost 训练: 全局直接多步 (单模型, horizon 为特征) + 线性目标
# 线性 MSE 直接对齐线性 RMSE, 峰值误差按量级放大 (=内建高值权重)
# 参考: https://xgboost.readthedocs.io/en/stable/parameter.html
#       https://xgboost.readthedocs.io/en/stable/python/python_api.html
# ============================================================
dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=FEAT)
dva = xgb.DMatrix(X_va, label=y_va, feature_names=FEAT)
params = {
    'objective':'reg:squarederror',
    'eval_metric':'rmse',
    'tree_method':'hist',
    'max_depth':8,
    'eta':0.05,
    'subsample':0.85,'colsample_bytree':0.85,
    'min_child_weight':3,'gamma':0.0,'lambda':0.5,'alpha':0.1,
    'nthread':4,
}
bst = xgb.train(params, dtr, num_boost_round=600,
                evals=[(dtr,'train'),(dva,'eval')],
                early_stopping_rounds=60, verbose_eval=50)
print(f'\n最佳轮次: {bst.best_iteration} | val RMSE (ug/m3): {bst.best_score:.2f}')

fig, ax = plt.subplots(figsize=(10, 6))
try:
    xgb.plot_importance(bst, max_num_features=N_FEAT, ax=ax)
    plt.title(f'Feature Importance ({N_FEAT} features)'); plt.tight_layout(); plt.show()
except Exception as e:
    plt.close(fig); print(f'[特征重要性绘图跳过] {e}')

In [ ]:
# ============================================================
# 评估: 36 组, 每组 24h 上下文@T + 未来气象 → 一次预测 48h (线性, 无递归)
# 指标: 分组平均 RMSE(48h|24h) + 池化综合指标 (MAE/FAC2/R/峰值RMSE·MAE)
# 并内置"持久性基线"同窗对比
# 参考: https://xgboost.readthedocs.io/en/stable/tutorials/predicting.html
# ============================================================
N_GROUPS, N_CTX, N_PRED = 36, 24, 48
STRIDE = (len(df_te) - N_CTX - N_PRED) // (N_GROUPS - 1)
hist_te = hist_matrix(df_te); fut_te = df_te[FUT_COLS].values
pm_te = np.expm1(df_te['pm_ave'].values)
PEAK = 75.0   # 峰值阈值 (ug/m3)

def predict_groups():
    P, A = [], []
    for g in range(N_GROUPS):
        T = g * STRIDE + N_CTX
        Xrow = np.empty((N_PRED, N_FEAT))
        Xrow[:, :NH] = hist_te[T]
        Xrow[:, NH:NH+NF] = fut_te[T:T+N_PRED]
        Xrow[:, NH+NF] = np.arange(1, N_PRED+1)
        P.append(np.clip(bst.predict(xgb.DMatrix(Xrow, feature_names=FEAT)), 0, CLIP_HI))
        A.append(pm_te[T:T+N_PRED])
    return P, A

def persist_groups():
    """持久性基线: 未来48h常数 = 最近一小时观测值"""
    P, A = [], []
    for g in range(N_GROUPS):
        T = g * STRIDE + N_CTX
        P.append(np.full(N_PRED, pm_te[T-1])); A.append(pm_te[T:T+N_PRED])
    return P, A

def group_rmse(P, A, k=None):
    return float(np.mean([np.sqrt(((p[:k]-a[:k])**2).mean()) for p, a in zip(P, A)]))

def pooled(P, A, k, tag):
    p = np.concatenate([x[:k] for x in P]); a = np.concatenate([x[:k] for x in A])
    rmse = np.sqrt(((p-a)**2).mean()); mae = np.abs(p-a).mean()
    fac2 = np.mean((p/(a+1e-6) <= 2) & (a/(p+1e-6) <= 2)); r = np.corrcoef(p, a)[0,1]
    pk = a > PEAK
    prmse = np.sqrt(((p[pk]-a[pk])**2).mean()) if pk.sum() else float('nan')
    pmae = np.abs(p[pk]-a[pk]).mean() if pk.sum() else float('nan')
    print(f'  [{tag}] RMSE={rmse:.2f} MAE={mae:.2f} FAC2={fac2:.3f} R={r:.3f} | 峰值(>{PEAK:.0f})n={int(pk.sum())} RMSE={prmse:.2f} MAE={pmae:.2f}')

P, A = predict_groups()
rmses = [np.sqrt(((p-a)**2).mean()) for p, a in zip(P, A)]
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, A[g], lw=0.8, label='actual'); ax.plot(h, P[g], lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[g]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'XGBoost Direct (multi-station) — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

PP, AA = persist_groups()
print('=== 分组平均 RMSE (既定基准) ===')
print(f'  XGBoost(多站) : 48h={group_rmse(P,A):.2f} | 24h={group_rmse(P,A,24):.2f}')
print(f'  持久性        : 48h={group_rmse(PP,AA):.2f} | 24h={group_rmse(PP,AA,24):.2f}')
print('\n=== 池化综合指标 (XGBoost 多站) ===')
pooled(P, A, 48, '48h'); pooled(P, A, 24, '24h')
print('=== 池化综合指标 (持久性) ===')
pooled(PP, AA, 48, '48h'); pooled(PP, AA, 24, '24h')
print('\n=== 各组 RMSE(48h|24h) ===')
for g in range(N_GROUPS):
    r48 = np.sqrt(((P[g]-A[g])**2).mean()); r24 = np.sqrt(((P[g][:24]-A[g][:24])**2).mean())
    print(f'  G{g+1:2d}: {r48:6.2f} | {r24:6.2f}')